# 🧠 Train your classifier (Week 6 — Classical ML)
**The trick (transfer learning):** CLIP was already trained on 400 million images. We use it to turn each photo into **512 numbers** that describe it.
Then we train a simple **classical ML** model (logistic regression) on those numbers. That's it — it trains in seconds.

> Turn on a GPU for speed: Runtime → Change runtime type → T4 GPU (optional; CPU works too).

In [ ]:
# 👉 Change this to YOUR GitHub username, then run every cell (Runtime → Run all)
GITHUB_USERNAME = "your-username"

import os
repo = f"https://github.com/{GITHUB_USERNAME}/snapsnipe"
if os.path.exists("snapsnipe"):
    !cd snapsnipe && git pull -q
else:
    !git clone -q {repo}
%cd snapsnipe
!pip install -q -r requirements.txt

## 1. Turn every photo into 512 numbers

In [ ]:
from pathlib import Path
import numpy as np
from config import CATEGORIES
from model_utils import embed

paths, labels = [], []
for cat in CATEGORIES:
    for p in sorted(Path("data/raw", cat).glob("*")):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            paths.append(str(p)); labels.append(cat)
labels = np.array(labels)

X = embed(paths)
print("Features:", X.shape, "→ one row per photo, 512 numbers each")

## 2. Hold back some photos for testing
The model never sees the *test* photos while training — so the test score is honest.

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(paths))
idx_train, idx_test = train_test_split(idx, test_size=0.25, stratify=labels, random_state=42)
X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = labels[idx_train], labels[idx_test]
print(f"train: {len(idx_train)} photos   test: {len(idx_test)} photos")

## 3. Compare 3 models

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from model_utils import zero_shot

results = {}

# A) Zero-shot: no training at all, just category names
zs_preds = [max(zero_shot(paths[i], CATEGORIES).items(), key=lambda kv: kv[1])[0] for i in idx_test]
results["Zero-shot CLIP (no training)"] = accuracy_score(y_test, zs_preds)

# B) k-nearest neighbors: "what do the 5 most similar photos say?"
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
results["k-nearest neighbors"] = accuracy_score(y_test, knn.predict(X_test))

# C) Logistic regression: learns a weight for each of the 512 numbers
clf = LogisticRegression(max_iter=2000).fit(X_train, y_train)
results["Logistic regression"] = accuracy_score(y_test, clf.predict(X_test))

table = pd.DataFrame({"model": results.keys(), "test accuracy": [f"{v:.0%}" for v in results.values()]})
print(table.to_markdown(index=False))   # 👉 paste this into your README

## 4. Where does it get confused?
Rows = the real answer, columns = what the model guessed. Off-diagonal = mistakes.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

preds = clf.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, preds, labels=CATEGORIES, cmap="Blues")
plt.title("Confusion matrix"); plt.show()
print(classification_report(y_test, preds, labels=CATEGORIES, zero_division=0))

## 5. Look at every mistake 🕵️
For each one, write down *why* you think it happened.

In [ ]:
from PIL import Image

wrong = [(paths[i], y, p) for i, y, p in zip(idx_test, y_test, preds) if y != p]
print(f"{len(wrong)} mistakes out of {len(y_test)}")
if wrong:
    cols = 4
    rows = (len(wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, (path, true, guess) in zip(axes.flat, wrong):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(f"real: {true}\nguess: {guess}", fontsize=9, color="crimson")
        print(path)
    plt.tight_layout(); plt.show()

## 6. Save your model 💾
This retrains on **all** your photos (more data = better), then saves `model.joblib`.

Download it (📁 Files panel on the left → `snapsnipe/model.joblib` → ⋮ → Download), then upload it to the **root** of your GitHub repo. Your app will use it automatically.

In [ ]:
import joblib
final = LogisticRegression(max_iter=2000).fit(X, labels)
joblib.dump({"clf": final, "categories": CATEGORIES}, "model.joblib")
print("✅ Saved model.joblib")

try:
    from google.colab import files
    files.download("model.joblib")
except ImportError:
    pass

## 📣 Marketing bonus
Write your result like a product launch headline and put it at the top of your README:

> *"SnapSnipe tells a pizza from a burrito **94%** of the time."*